# Molmo 2: Can a Vision Model Point to Hydraulic Features?

**Goal:** Test whether Molmo 2 can go beyond *describing* features (like Gemma 4 did) and actually *point to them* with pixel coordinates.

Molmo 2 is unique among vision-language models: it can output `(x, y)` coordinates in the image, not just text descriptions. This is the critical bridge to **SAM** (Stage 6) — if Molmo 2 can point to a river channel, we can feed those coordinates directly to SAM as prompts for precise segmentation.

| Model | Capability | Output |
|-------|-----------|--------|
| Gemma 4 | Describe features | "The river channel is in the center" |
| **Molmo 2** | **Point to features** | **`<point x="52.3" y="41.7">river channel</point>`** |
| SAM (Stage 6) | Segment from points | Binary mask of the river channel |

## Steps
1. Connect to Molmo 2 via OpenRouter
2. Test pointing on hillshade, slope, curvature, elevation
3. Parse coordinates and convert to geo-coordinates
4. Overlay points on terrain images
5. Compare qualitative descriptions with Gemma 4 results

In [ ]:
import os
import base64
import io
import re
import json
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import rasterio
from PIL import Image
from openai import OpenAI

# Paths
DEM_PATH = Path("../data/input/1m elevation.tif")
OUTPUT_DIR = Path("../data/output")
RESULTS_DIR = Path("../docs/model-outputs/molmo2")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"DEM: {DEM_PATH}")
print(f"Output images: {OUTPUT_DIR}")
print(f"Results: {RESULTS_DIR}")

## 1.1 Connect to Molmo 2 via OpenRouter

In [ ]:
# Load .env file if python-dotenv is available
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass  # Set OPENROUTER_API_KEY in your environment manually

OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "")
if not OPENROUTER_API_KEY:
    raise ValueError(
        "Set OPENROUTER_API_KEY in your .env file or environment.\n"
        "Get a free key at: https://openrouter.ai/keys"
    )

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)

MODEL = "allenai/molmo-2-8b"
MOLMO_AVAILABLE = False

# Test connection
try:
    test = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": "Say hello in one word."}],
        max_tokens=10,
    )
    print(f"Molmo 2 connection OK: {test.choices[0].message.content}")
    print(f"Model: {MODEL}")
    MOLMO_AVAILABLE = True
except Exception as e:
    print(f"Molmo 2 is NOT available on OpenRouter.")
    print(f"Error: {e}")
    print()
    print("This is expected — Molmo 2 has a page on OpenRouter but may not be")
    print("serving API requests yet. Try these alternatives:")
    print("  - allenai/molmo-7b-d-0924  (Molmo 1, may still work)")
    print("  - Run Molmo 2 locally via Ollama or vLLM")
    print("  - Check https://openrouter.ai/models for current availability")
    print()
    print("The rest of this notebook will be skipped.")

## 1.2 Load DEM Data

In [ ]:
# Load DEM metadata — needed for geo-coordinate conversion
with rasterio.open(DEM_PATH) as src:
    dem_transform = src.transform
    dem_bounds = src.bounds
    dem_crs = src.crs

# Extent for matplotlib (left, right, bottom, top)
extent = [dem_bounds.left, dem_bounds.right, dem_bounds.bottom, dem_bounds.top]

print(f"CRS: {dem_crs}")
print(f"Bounds: {dem_bounds}")

# Load pre-exported PNG images from data/output/
terrain_images = {}
for name in ["hillshade", "slope", "curvature", "elevation"]:
    path = OUTPUT_DIR / f"cimarron_{name}.png"
    if path.exists():
        terrain_images[name] = Image.open(path)
        print(f"Loaded {name}: {terrain_images[name].size}")
    else:
        print(f"WARNING: {path} not found — run notebook 01 first")

print(f"\nLoaded {len(terrain_images)} terrain images.")

## 1.3 Inference & Coordinate Parsing Utilities

In [ ]:
def resize_for_api(img, max_width=1024):
    """Resize image to reduce API token usage while preserving aspect ratio."""
    if img.width > max_width:
        ratio = max_width / img.width
        new_size = (max_width, int(img.height * ratio))
        return img.resize(new_size, Image.LANCZOS)
    return img


def image_to_base64(img):
    """Convert PIL Image to base64 data URL for the API."""
    img = resize_for_api(img)
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    b64 = base64.b64encode(buf.getvalue()).decode("utf-8")
    return f"data:image/png;base64,{b64}"


def vision_query(image, prompt, max_tokens=512):
    """Send an image + text prompt to the vision model and return the response."""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                    {"type": "image_url", "image_url": {"url": image_to_base64(image)}},
                ],
            }
        ],
        max_tokens=max_tokens,
    )
    return response.choices[0].message.content


def parse_molmo_points(text, image_w, image_h):
    """Parse Molmo coordinate output into pixel coordinates.

    Handles both formats:
    - Legacy (0-100 scale): <point x="10.0" y="20.0">label</point>
    - Molmo 2 (x1000 scale): <points coords="x1,y1,x2,y2,..."/>

    Returns list of (x_px, y_px, label) tuples.
    """
    points = []

    # Legacy format: <point x="..." y="...">label</point>
    legacy_pattern = r'<point\s+x="([\d.]+)"\s+y="([\d.]+)"(?:\s+alt="[^"]*")?>([^<]*)</point>'
    for match in re.finditer(legacy_pattern, text):
        x_pct, y_pct, label = float(match.group(1)), float(match.group(2)), match.group(3).strip()
        # Convert 0-100 scale to pixel coordinates
        x_px = x_pct / 100.0 * image_w
        y_px = y_pct / 100.0 * image_h
        points.append((x_px, y_px, label))

    # Molmo 2 format: <points coords="x1,y1,x2,y2,..."/>
    coords_pattern = r'<points\s+coords="([^"]+)"\s*/>'
    for match in re.finditer(coords_pattern, text):
        coords = [float(c) for c in match.group(1).split(",")]
        for i in range(0, len(coords) - 1, 2):
            # x1000 scale: divide by 1000 then multiply by image dimensions
            x_px = coords[i] / 1000.0 * image_w
            y_px = coords[i + 1] / 1000.0 * image_h
            points.append((x_px, y_px, f"point_{i // 2}"))

    return points


def pixels_to_geo(points, transform):
    """Convert pixel coordinates to geo-coordinates using the rasterio transform.

    Returns list of (easting, northing) tuples.
    """
    geo_coords = []
    for x_px, y_px, label in points:
        easting, northing = rasterio.transform.xy(transform, int(y_px), int(x_px))
        geo_coords.append((easting, northing, label))
    return geo_coords


def molmo_point(image, prompt):
    """Run a pointing query and parse the results.

    Returns dict with raw_output, pixel_coords, geo_coords, n_points.
    """
    resized = resize_for_api(image)
    raw = vision_query(image, prompt)
    pixel_coords = parse_molmo_points(raw, resized.width, resized.height)
    geo_coords = pixels_to_geo(pixel_coords, dem_transform) if pixel_coords else []
    return {
        "raw_output": raw,
        "pixel_coords": pixel_coords,
        "geo_coords": geo_coords,
        "n_points": len(pixel_coords),
    }


def plot_points_on_terrain(terrain_img, points_px, title, figsize=(14, 8)):
    """Overlay parsed points on a terrain image."""
    fig, ax = plt.subplots(1, 1, figsize=figsize)
    ax.imshow(terrain_img)
    ax.set_title(title, fontsize=14)

    if points_px:
        xs = [p[0] for p in points_px]
        ys = [p[1] for p in points_px]
        labels = [p[2] for p in points_px]
        ax.scatter(xs, ys, c="red", s=80, marker="x", linewidths=2, zorder=5)
        for x, y, label in zip(xs, ys, labels):
            ax.annotate(label, (x, y), textcoords="offset points",
                        xytext=(5, 5), fontsize=8, color="red",
                        bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.7))
        ax.set_xlabel(f"{len(points_px)} points detected")
    else:
        ax.set_xlabel("No coordinate points parsed from model output")

    ax.set_xticks([])
    ax.set_yticks([])
    plt.tight_layout()
    return fig


# Store all results
vision_results = {}
pointing_results = {}

print("Utilities ready.")

## 2.1 Pointing Tests — Hillshade

In [ ]:
if not MOLMO_AVAILABLE:
    print("Skipping — Molmo 2 is not available.")
else:
    img_hs = terrain_images["hillshade"]

    hillshade_pointing_prompts = {
        "channel": "Point to the main river channel in this terrain image.",
        "ridges": "Point to the ridges or terrace edges in this terrain image.",
        "floodplain": "Point to the flat floodplain areas in this terrain image.",
        "manmade": "Point to any road embankments or linear man-made features in this terrain image.",
    }

    for key, prompt in hillshade_pointing_prompts.items():
        test_name = f"hillshade_{key}"
        print(f"\n{'='*80}")
        print(f"TEST: {test_name}")
        print(f"Prompt: {prompt}")
        print(f"{'='*80}")

        result = molmo_point(img_hs, prompt)
        pointing_results[test_name] = {"prompt": prompt, **result}

        print(f"\nRaw output:\n{result['raw_output']}")
        print(f"\nPoints found: {result['n_points']}")
        if result['pixel_coords']:
            for px, py, label in result['pixel_coords']:
                print(f"  ({px:.0f}, {py:.0f}) — {label}")

        fig = plot_points_on_terrain(img_hs, result['pixel_coords'],
                                     f"Molmo 2 Pointing: {key} (hillshade)")
        plt.show()

## 2.2 Pointing Tests — Alternative Renderings

Test the channel-pointing prompt on slope, curvature, and elevation to see which rendering works best for coordinate extraction.

In [ ]:
if not MOLMO_AVAILABLE:
    print("Skipping — Molmo 2 is not available.")
else:
    channel_prompt = "Point to the main river channel in this terrain image."

    for rendering in ["slope", "curvature", "elevation"]:
        test_name = f"{rendering}_channel"
        print(f"\n{'='*80}")
        print(f"TEST: {test_name}")
        print(f"{'='*80}")

        img = terrain_images[rendering]
        result = molmo_point(img, channel_prompt)
        pointing_results[test_name] = {"prompt": channel_prompt, **result}

        print(f"\nRaw output:\n{result['raw_output']}")
        print(f"\nPoints found: {result['n_points']}")
        if result['pixel_coords']:
            for px, py, label in result['pixel_coords']:
                print(f"  ({px:.0f}, {py:.0f}) — {label}")

        fig = plot_points_on_terrain(img, result['pixel_coords'],
                                     f"Molmo 2 Pointing: channel ({rendering})")
        plt.show()

## 2.3 Qualitative Comparison

Run the same general survey prompt used with Gemma 4 to allow direct comparison of descriptive capabilities.

In [ ]:
if not MOLMO_AVAILABLE:
    print("Skipping — Molmo 2 is not available.")
else:
    survey_prompt = (
        "This is a hillshade rendering of a 1-meter resolution DEM (digital elevation model) "
        "of a river corridor. Identify all hydraulic and geomorphic features you can see. "
        "For each feature, describe its type, approximate location in the image, and shape. "
        "Look for: river channels, floodplains, terrace edges, ridges, point bars, oxbow lakes, "
        "levees, road embankments, or any other man-made structures."
    )

    img_hs = terrain_images["hillshade"]

    print(f"{'='*80}")
    print(f"TEST: qualitative_survey (same prompt as Gemma 4)")
    print(f"{'='*80}")

    response = vision_query(img_hs, survey_prompt, max_tokens=1024)
    vision_results["qualitative_survey"] = {
        "prompt": survey_prompt,
        "response": response,
        "rendering": "hillshade",
    }

    print(response)

## 3. Results Summary

In [ ]:
if not MOLMO_AVAILABLE:
    print("Molmo 2 was not available — no results to summarize.")
    print("Update MODEL in cell 4 to test an alternative, or try again later.")
else:
    # --- Pointing results table ---
    print("POINTING RESULTS")
    print(f"{'Test':<25} {'Points':>6}  {'Coordinates (pixel)'}")
    print("-" * 80)
    for key, result in pointing_results.items():
        n = result["n_points"]
        if result["pixel_coords"]:
            coords_str = "; ".join(f"({p[0]:.0f},{p[1]:.0f})" for p in result["pixel_coords"][:5])
            if n > 5:
                coords_str += f" ... (+{n - 5} more)"
        else:
            coords_str = "(no coordinates parsed)"
        print(f"{key:<25} {n:>6}  {coords_str}")

    # --- Qualitative results table ---
    print(f"\n\nQUALITATIVE RESULTS")
    print(f"{'Test':<25} {'Length':>8}  {'Preview'}")
    print("-" * 80)
    for key, result in vision_results.items():
        resp_len = len(result["response"])
        preview = result["response"][:60].replace("\n", " ")
        print(f"{key:<25} {resp_len:>5} ch  {preview}...")

    print(f"\nModel: {MODEL}")
    print(f"Total pointing tests: {len(pointing_results)}")
    print(f"Total qualitative tests: {len(vision_results)}")

    # --- Save results to markdown ---
    md_lines = [
        "# Molmo 2 — DEM Pointing Test Results\n",
        f"**Date:** April 8, 2026\n",
        f"**Model:** `{MODEL}` (via OpenRouter)\n",
        f"**DEM:** Cimarron River, 1m resolution, {dem_crs}\n",
        f"**Images:** Resized to 1024px wide before sending to API\n",
        f"**Notebook:** `notebooks/02_molmo2_pointing_test.ipynb`\n",
        "\n---\n",
    ]

    # Pointing tests
    for key, result in pointing_results.items():
        md_lines.append(f"## Pointing: {key.replace('_', ' ').title()}\n")
        md_lines.append(f"**Prompt:** {result['prompt']}\n")
        md_lines.append(f"**Points found:** {result['n_points']}\n")
        if result["pixel_coords"]:
            md_lines.append("\n**Pixel coordinates:**\n")
            for px, py, label in result["pixel_coords"]:
                md_lines.append(f"- ({px:.0f}, {py:.0f}) — {label}\n")
        if result["geo_coords"]:
            md_lines.append("\n**Geo-coordinates (EPSG:26914):**\n")
            for e, n, label in result["geo_coords"]:
                md_lines.append(f"- ({e:.1f}, {n:.1f}) — {label}\n")
        md_lines.append(f"\n**Raw output:**\n\n{result['raw_output']}\n")
        md_lines.append("\n---\n")

    # Qualitative tests
    for key, result in vision_results.items():
        md_lines.append(f"## {key.replace('_', ' ').title()}\n")
        md_lines.append(f"**Prompt:** {result['prompt']}\n")
        md_lines.append(f"\n**Response:**\n\n{result['response']}\n")
        md_lines.append("\n---\n")

    # Summary table
    md_lines.append("## Summary\n")
    md_lines.append("| Test | Points | Notes |\n")
    md_lines.append("|------|--------|-------|\n")
    for key, result in pointing_results.items():
        n = result["n_points"]
        note = "coordinates parsed" if n > 0 else "no coordinates in output"
        md_lines.append(f"| {key} | {n} | {note} |\n")

    output_path = RESULTS_DIR / "2026-04-08-molmo2-pointing-test.md"
    with open(output_path, "w") as f:
        f.writelines(md_lines)

    print(f"\nResults saved to: {output_path}")